# Notebook 04 — A Home for Your Vectors (ChromaDB)

> **Easiest way to run this: Google Colab — nothing to install.**
> Go to https://colab.research.google.com → **File > Upload notebook** → choose this file.
> Prefer your own computer? Lesson 1 shows the VS Code and local-Jupyter paths too.

In Notebook 03 we found the closest sentence by comparing a query against **all twelve**
sentences by hand. Twelve is fine. A million is not — you don't want to write that loop, and
you don't want it slow. A **vector store** is a tool whose whole job is: hold a pile of
vectors, and instantly find the ones nearest a query.

> **Analogy:** a library with a magic catalogue. You describe what you want; it walks you
> straight to the closest shelf — without you scanning every book.

We'll use **ChromaDB**: free, local, nothing leaves your machine.

In [ ]:
%pip install -q sentence-transformers chromadb
print("Ready.")

## A quick primer on ChromaDB

A few words first, so the code reads clearly:

- A **database** is just an organised store of information you can put things into and get
  back out of.
- A **vector store** (or **vector database**) is a database built for *vectors*
  (embeddings). Its special skill is finding the stored vectors **nearest** to a query —
  the meaning-search from Notebook 01, done for you, and fast even with millions of items.
- A **collection** is one named group of items inside the store — like a single table, or
  one labelled drawer. You make one with `client.create_collection(name=...)`.
- The **documents** are the pieces of text you store. ChromaDB embeds each one for you
  (with MiniLM), so you hand it plain text, not numbers.
- Each item needs a unique **id** (a name tag like `"s0"`, `"s1"`) so the store can tell
  items apart.
- **metadata** is optional extra information you attach to an item (later we tuck an
  answer next to a question this way).
- A **query** is how you search: you give `collection.query(...)` some text, and it hands
  back the nearest stored documents.
- **distance / space**: we set the store to `"cosine"` space, so "closeness" means the
  cosine similarity you already know. ChromaDB reports a **distance**, and
  `similarity = 1 - distance`.

That is the whole vocabulary. Now let's build one.

## Step 1 — Create a store that speaks MiniLM

We tell ChromaDB to use the **same** MiniLM model from Notebook 03 to turn text into
vectors. Then every sentence we add is embedded automatically — we just hand it text. We also
set the store to measure closeness by **cosine** (angle), the same "pointing the same way"
notion we've used all along.

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

minilm_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# A store saved to a folder on disk, so it survives after the notebook closes.
client = chromadb.PersistentClient(path="./chroma_store")

# Start fresh each run so re-running doesn't pile up duplicates.
try:
    client.delete_collection("sentences")
except Exception:
    pass

collection = client.create_collection(
    name="sentences",
    embedding_function=minilm_ef,
    metadata={"hnsw:space": "cosine"},
)
print("Empty collection created (it lives in the ./chroma_store folder).")

## Step 2 — Add the twelve sentences

We hand ChromaDB plain text plus an id for each. It embeds them with MiniLM behind the
scenes and stores the vectors. No manual `encode` call needed.

In [ ]:
sentences = [
    "The dog wagged its tail when its owner came home.",
    "A kitten chased a ball of yarn across the floor.",
    "Lions live in prides on the African savanna.",
    "The parrot mimicked every word the children said.",
    "I drove the car to the grocery store this morning.",
    "The new electric bicycle has a range of sixty miles.",
    "Trucks deliver packages to our neighbourhood every day.",
    "The airplane landed smoothly despite the strong winds.",
    "She baked sourdough bread for the first time on Sunday.",
    "The pizza was hot and covered in melted mozzarella.",
    "I ordered sushi for lunch at the new Japanese restaurant.",
    "He grilled steak and roasted vegetables for dinner.",
]
ids = [f"sent_{i}" for i in range(len(sentences))]

collection.add(ids=ids, documents=sentences)
print(f"Stored {collection.count()} sentences.")

## Step 3 — Ask it a question

`collection.query` embeds your query, finds the nearest stored vectors, and hands back the
matching sentences — the same nearest-neighbour search from Notebook 01, now done for us at
any scale.

In [ ]:
result = collection.query(
    query_texts=["something tasty to eat"],
    n_results=3,
)

print("Query: 'something tasty to eat'\n")
for doc, dist in zip(result["documents"][0], result["distances"][0]):
    # With cosine space, similarity = 1 - distance. Higher = closer in meaning.
    print(f"  similarity {1 - dist:.3f}   {doc}")

### What to notice
- The top results are the **food** sentences — even though the query never says "pizza" or
  "sushi". Meaning, not keywords.
- ChromaDB returns a **distance**; with cosine space, `similarity = 1 - distance`. Bigger
  similarity = closer in meaning.
- The store sits in the `./chroma_store` folder. Close the notebook, reopen, and the data is
  still there — no need to re-embed.

**Try it:** change the query to "a wild animal" or "ways to get around town" and re-run.

## Recap

- A vector store holds many vectors and finds the nearest ones to a query — fast, at any
  scale.
- ChromaDB embeds text for us (with MiniLM), stores it locally, and searches it.
- `add` puts text in; `query` pulls the closest text out. That's the whole interface.
- Cosine distance and cosine similarity are two sides of one coin: `sim = 1 - dist`.

**Next (Notebook 05):** we wire this into a tiny **semantic search engine** over a small
document collection — the thing promised in Lesson 0.